# Image Segmentation

## Learning Objectives
1. Implement Dice loss from scratch and understand why it handles class imbalance
2. Build a U-Net with skip connections in PyTorch
3. Compare Dice loss vs cross-entropy on imbalanced segmentation data
4. Implement morphological post-processing and multi-class IoU computation


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import jaccard_score

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## Level 1: Dice Loss from Scratch + Manual Segmentation

In [ ]:
# ---- Level 1: Dice loss and binary mask metrics in numpy -----------------
# Dice coefficient = 2|A intersect B| / (|A| + |B|)
# It equals F1 score at the pixel level and is invariant to class frequency.


def dice_coefficient(pred_mask, true_mask, smooth=1e-6):
    """Compute Dice coefficient between two binary masks.

    Args:
        pred_mask: numpy array of 0/1 predictions (any shape)
        true_mask: numpy array of 0/1 ground truth
        smooth: small constant prevents division by zero on empty masks

    Returns:
        Dice coefficient in [0, 1]. 1.0 = perfect match.
    """
    pred_flat = pred_mask.flatten().astype(float)
    true_flat = true_mask.flatten().astype(float)
    intersection = np.sum(pred_flat * true_flat)  # logical AND
    return (2.0 * intersection + smooth) / (np.sum(pred_flat) + np.sum(true_flat) + smooth)


def iou_coefficient(pred_mask, true_mask, smooth=1e-6):
    """Compute Intersection over Union (Jaccard index) for binary masks.

    Related to Dice: IoU = Dice / (2 - Dice)
    """
    pred_flat = pred_mask.flatten().astype(float)
    true_flat = true_mask.flatten().astype(float)
    intersection = np.sum(pred_flat * true_flat)
    union = np.sum(pred_flat) + np.sum(true_flat) - intersection
    return (intersection + smooth) / (union + smooth)


def pixel_accuracy(pred_mask, true_mask):
    """Fraction of pixels correctly classified.

    NOTE: misleading on class-imbalanced data — use Dice or IoU instead.
    """
    return np.mean(pred_mask == true_mask)


# Demonstrate on a 8x8 toy example
# Ground truth: a 3x3 square in the center
gt = np.zeros((8, 8))
gt[2:5, 3:6] = 1

# Case 1: Perfect prediction
pred_perfect = gt.copy()
# Case 2: Slight shift (one pixel off)
pred_shifted = np.zeros((8, 8))
pred_shifted[3:6, 4:7] = 1
# Case 3: All zeros (background only — the imbalanced model failure mode)
pred_all_bg = np.zeros((8, 8))

print('Metric comparison on 8x8 toy segmentation (foreground is 9/64 pixels):')
print(f'               Pixel Accuracy  Dice    IoU')
for pred, name in [
    (pred_perfect, 'Perfect        '),
    (pred_shifted, 'Shifted 1px    '),
    (pred_all_bg,  'All background '),
]:
    pa = pixel_accuracy(pred, gt)
    dc = dice_coefficient(pred, gt)
    iou = iou_coefficient(pred, gt)
    print(f'{name}:  PA={pa:.2f}          Dice={dc:.2f}  IoU={iou:.2f}')

print('\nKey insight: All-background has 86% pixel accuracy but 0 Dice and 0 IoU.')
print('Pixel accuracy is useless on imbalanced segmentation — always use Dice/IoU.')

# Visualize the three cases
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, data, title in zip(
    axes,
    [gt, pred_perfect, pred_shifted, pred_all_bg],
    ['Ground Truth', 'Perfect', 'Shifted 1px', 'All Background'],
):
    ax.imshow(data, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.suptitle('Level 1: Segmentation Metrics on Toy Data')
plt.tight_layout()
plt.savefig('/tmp/cv03_metrics.png', dpi=80)
plt.close()
print('Saved: /tmp/cv03_metrics.png')


## Level 2: U-Net with Skip Connections in PyTorch

In [ ]:
# ---- Level 2: U-Net with skip connections for synthetic circle detection --
# The encoder compresses the image; decoder uses skip connections to recover
# fine-grained boundary locations lost during downsampling.


class DoubleConv(nn.Module):
    """Two consecutive ConvBNReLU blocks — the basic unit in U-Net."""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """U-Net for binary segmentation.

    Architecture:
    - Encoder: 3 DoubleConv + MaxPool stages (32->64->128 channels)
    - Bottleneck: deepest representation (256 channels)
    - Decoder: 3 upsample + skip concat + DoubleConv stages
    - Head: 1x1 conv to 1 output channel (binary mask)
    """

    def __init__(self, in_ch=1, out_ch=1, base_ch=32):
        super().__init__()
        # Encoder
        self.enc1 = DoubleConv(in_ch, base_ch)          # 64x64
        self.enc2 = DoubleConv(base_ch, base_ch * 2)    # 32x32
        self.enc3 = DoubleConv(base_ch * 2, base_ch * 4)  # 16x16
        self.pool = nn.MaxPool2d(2)
        # Bottleneck: smallest spatial, most channels
        self.bottleneck = DoubleConv(base_ch * 4, base_ch * 8)  # 8x8
        # Decoder: upsample + concatenate skip + DoubleConv
        # After concat, in_ch doubles because we append skip connection features
        self.up3 = nn.ConvTranspose2d(base_ch * 8, base_ch * 4, 2, stride=2)
        self.dec3 = DoubleConv(base_ch * 8, base_ch * 4)  # 16x16 (concat skip)
        self.up2 = nn.ConvTranspose2d(base_ch * 4, base_ch * 2, 2, stride=2)
        self.dec2 = DoubleConv(base_ch * 4, base_ch * 2)  # 32x32 (concat skip)
        self.up1 = nn.ConvTranspose2d(base_ch * 2, base_ch, 2, stride=2)
        self.dec1 = DoubleConv(base_ch * 2, base_ch)      # 64x64 (concat skip)
        # Final 1x1 conv — produces raw logit per pixel
        self.head = nn.Conv2d(base_ch, out_ch, 1)

    def forward(self, x):
        # Encoder with saved skip connection tensors
        s1 = self.enc1(x)           # saved for skip
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        # Bottleneck
        b = self.bottleneck(self.pool(s3))
        # Decoder: upsample + concatenate skip from matching encoder level
        d3 = self.dec3(torch.cat([self.up3(b), s3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), s2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), s1], dim=1))
        return self.head(d1)  # (B, 1, H, W) raw logits


class DiceLoss(nn.Module):
    """Differentiable Dice loss for binary segmentation.

    Uses sigmoid probabilities (not binary predictions) so the loss is
    differentiable and can be minimized via gradient descent.
    """

    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)  # convert logits to probabilities
        probs_flat = probs.view(-1)
        targets_flat = targets.view(-1)
        intersection = (probs_flat * targets_flat).sum()
        return 1 - (2.0 * intersection + self.smooth) / (
            probs_flat.sum() + targets_flat.sum() + self.smooth
        )


def generate_circle_masks(n=600, img_size=64):
    """Generate synthetic images with circle masks for U-Net training.

    Each image has 1-3 circles with random center, radius, and intensity.
    Returns images (N, 1, H, W) float and masks (N, 1, H, W) binary.
    """
    images = np.zeros((n, 1, img_size, img_size), dtype=np.float32)
    masks = np.zeros((n, 1, img_size, img_size), dtype=np.float32)
    yy, xx = np.mgrid[:img_size, :img_size]
    for i in range(n):
        n_circles = np.random.randint(1, 4)
        for _ in range(n_circles):
            cx = np.random.randint(10, img_size - 10)
            cy = np.random.randint(10, img_size - 10)
            r = np.random.randint(4, 14)
            intensity = np.random.uniform(0.5, 1.0)
            circle = (xx - cx)**2 + (yy - cy)**2 < r**2
            images[i, 0] += circle.astype(float) * intensity
            masks[i, 0] = np.clip(masks[i, 0] + circle.astype(float), 0, 1)
        images[i] = np.clip(images[i] + np.random.randn(1, img_size, img_size) * 0.1,
                            0, 1)
    return images, masks


# Build dataset
X_seg, y_seg = generate_circle_masks(n=600)
X_t = torch.tensor(X_seg)
y_t = torch.tensor(y_seg)
seg_ds = TensorDataset(X_t[:500], y_t[:500])
seg_val_ds = TensorDataset(X_t[500:], y_t[500:])
seg_loader = DataLoader(seg_ds, batch_size=32, shuffle=True)
seg_val_loader = DataLoader(seg_val_ds, batch_size=32)

# Train U-Net with Dice loss
unet = UNet(in_ch=1, out_ch=1).to(device)
unet_opt = optim.Adam(unet.parameters(), lr=1e-3)
dice_crit = DiceLoss()
bce_crit = nn.BCEWithLogitsLoss()

unet_losses = []
UNET_EPOCHS = 25
for epoch in range(UNET_EPOCHS):
    unet.train()
    ep_loss = 0.0
    for X_b, y_b in seg_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        unet_opt.zero_grad()
        try:
            pred = unet(X_b)
            # Combine Dice (handles imbalance) + BCE (stabilizes early training)
            loss = 0.5 * dice_crit(pred, y_b) + 0.5 * bce_crit(pred, y_b)
            loss.backward()
            unet_opt.step()
            ep_loss += loss.item()
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                print('OOM: reduce batch_size or model base_ch')
            else:
                raise
    unet_losses.append(ep_loss / len(seg_loader))

# Evaluate on validation set
unet.eval()
val_dices = []
with torch.no_grad():
    for X_v, y_v in seg_val_loader:
        pred_v = torch.sigmoid(unet(X_v.to(device)))
        pred_bin = (pred_v > 0.5).float().cpu().numpy()
        for p, t in zip(pred_bin, y_v.numpy()):
            val_dices.append(dice_coefficient(p[0], t[0]))

print(f'U-Net val Dice: {np.mean(val_dices):.3f} (mean over {len(val_dices)} images)')
n_params = sum(p.numel() for p in unet.parameters())
print(f'U-Net params: {n_params:,}')


## Real-World Example 1: Dice Loss vs Cross-Entropy on Imbalanced Data

In [ ]:
# ---- RW1: Dice vs BCE-CE on highly imbalanced segmentation ---------------
# When background occupies 95% of pixels, BCE loss is dominated by correct
# background predictions, so the model learns to predict all-background.
# Dice loss directly optimizes overlap and is immune to class frequency.


def generate_sparse_masks(n=400, img_size=64, foreground_pct=0.05):
    """Generate highly imbalanced data: small foreground circles in large images.

    foreground_pct controls how sparse the positive class is.
    At 0.05, roughly 5% of pixels are foreground — simulating medical lesions.
    """
    images = np.zeros((n, 1, img_size, img_size), dtype=np.float32)
    masks = np.zeros((n, 1, img_size, img_size), dtype=np.float32)
    yy, xx = np.mgrid[:img_size, :img_size]
    # Target radius to achieve foreground_pct coverage
    target_area = int(img_size * img_size * foreground_pct)
    r = int(np.sqrt(target_area / np.pi))
    for i in range(n):
        cx = np.random.randint(r + 5, img_size - r - 5)
        cy = np.random.randint(r + 5, img_size - r - 5)
        circle = (xx - cx)**2 + (yy - cy)**2 < r**2
        images[i, 0, circle] = np.random.uniform(0.6, 1.0)
        images[i, 0] += np.random.randn(img_size, img_size).astype(np.float32) * 0.1
        images[i] = np.clip(images[i], 0, 1)
        masks[i, 0, circle] = 1.0
    return images, masks


X_sparse, y_sparse = generate_sparse_masks(n=400, foreground_pct=0.05)
fg_frac = y_sparse.mean()
print(f'Foreground fraction: {fg_frac:.3f} ({fg_frac*100:.1f}% of pixels)')

# Build small U-Nets for fair comparison
X_sp_t = torch.tensor(X_sparse)
y_sp_t = torch.tensor(y_sparse)
sp_loader = DataLoader(TensorDataset(X_sp_t[:320], y_sp_t[:320]),
                       batch_size=32, shuffle=True)
sp_val = TensorDataset(X_sp_t[320:], y_sp_t[320:])


def train_segmenter(loss_fn, tag, epochs=20):
    """Train a small U-Net with the given loss function; return per-epoch val Dice."""
    m = UNet(in_ch=1, out_ch=1, base_ch=16).to(device)
    opt = optim.Adam(m.parameters(), lr=1e-3)
    val_dices_per_epoch = []
    for epoch in range(epochs):
        m.train()
        for X_b, y_b in sp_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            opt.zero_grad()
            pred = m(X_b)
            loss = loss_fn(pred, y_b)
            loss.backward()
            opt.step()
        # Per-epoch val Dice
        m.eval()
        ep_dices = []
        with torch.no_grad():
            for X_v, y_v in DataLoader(sp_val, batch_size=32):
                pv = (torch.sigmoid(m(X_v.to(device))) > 0.5).float().cpu().numpy()
                for p, t in zip(pv, y_v.numpy()):
                    ep_dices.append(dice_coefficient(p[0], t[0]))
        val_dices_per_epoch.append(np.mean(ep_dices))
    print(f'{tag}: final Dice = {val_dices_per_epoch[-1]:.3f}')
    return val_dices_per_epoch


dice_loss_fn = DiceLoss()
bce_loss_fn = nn.BCEWithLogitsLoss()
# Weighted BCE: upweight positive class inversely to its frequency
pos_weight = torch.tensor([(1 - fg_frac) / fg_frac]).to(device)
bce_weighted_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

dice_curve = train_segmenter(dice_loss_fn, 'Dice Loss')
bce_curve = train_segmenter(bce_loss_fn, 'BCE (unweighted)')
bce_w_curve = train_segmenter(bce_weighted_fn, 'BCE (weighted)')

plt.figure(figsize=(9, 4))
plt.plot(dice_curve, label='Dice Loss', color='steelblue')
plt.plot(bce_curve, label='BCE (unweighted)', color='tomato', linestyle='--')
plt.plot(bce_w_curve, label='BCE (weighted)', color='orange', linestyle='-.')
plt.xlabel('Epoch')
plt.ylabel('Validation Dice')
plt.title('Loss Function Comparison on 5% Foreground Data')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/cv03_loss_comparison.png', dpi=80)
plt.close()
print('Saved: /tmp/cv03_loss_comparison.png')


## Real-World Example 2: Morphological Post-Processing

In [ ]:
# ---- RW2: Morphological post-processing — erosion, dilation, connected ----
# After sigmoid thresholding, predicted masks often contain small noise blobs
# and holes. Morphological operations clean these artifacts.


def erode(mask, kernel_size=3):
    """Binary erosion: shrinks foreground regions by kernel_size pixels.

    A pixel is kept as foreground only if all pixels in its kernel_size x
    kernel_size neighborhood are foreground. Removes small noise blobs.
    """
    pad = kernel_size // 2
    padded = np.pad(mask, pad, mode='constant', constant_values=0)
    H, W = mask.shape
    out = np.ones_like(mask)
    for i in range(H):
        for j in range(W):
            neighborhood = padded[i:i + kernel_size, j:j + kernel_size]
            # Keep as foreground only if entire neighborhood is foreground
            out[i, j] = 1 if neighborhood.min() == 1 else 0
    return out


def dilate(mask, kernel_size=3):
    """Binary dilation: expands foreground regions by kernel_size pixels.

    A pixel becomes foreground if ANY pixel in its neighborhood is foreground.
    Fills small holes and connects nearby components.
    """
    pad = kernel_size // 2
    padded = np.pad(mask, pad, mode='constant', constant_values=0)
    H, W = mask.shape
    out = np.zeros_like(mask)
    for i in range(H):
        for j in range(W):
            neighborhood = padded[i:i + kernel_size, j:j + kernel_size]
            out[i, j] = 1 if neighborhood.max() == 1 else 0
    return out


def opening(mask, kernel_size=3):
    """Morphological opening: erode then dilate.

    Removes small noise blobs (smaller than kernel) without shrinking
    larger foreground regions significantly.
    """
    return dilate(erode(mask, kernel_size), kernel_size)


def remove_small_components(mask, min_size=20):
    """Remove connected components smaller than min_size pixels.

    Uses flood-fill to find connected regions, then filters by area.
    This is the most effective post-processing step for removing noise blobs.
    """
    from collections import deque
    visited = np.zeros_like(mask)
    cleaned = np.zeros_like(mask)
    H, W = mask.shape

    def bfs(start_r, start_c):
        component = []
        queue = deque([(start_r, start_c)])
        visited[start_r, start_c] = 1
        while queue:
            r, c = queue.popleft()
            component.append((r, c))
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr, nc = r + dr, c + dc
                if 0<=nr<H and 0<=nc<W and mask[nr,nc]==1 and not visited[nr,nc]:
                    visited[nr, nc] = 1
                    queue.append((nr, nc))
        return component

    for i in range(H):
        for j in range(W):
            if mask[i, j] == 1 and not visited[i, j]:
                comp = bfs(i, j)
                # Keep only components larger than min_size
                if len(comp) >= min_size:
                    for r, c in comp:
                        cleaned[r, c] = 1
    return cleaned


# Build a noisy predicted mask to demonstrate post-processing
# Start with a clean circle mask and add random noise blobs
img_size = 64
yy, xx = np.mgrid[:img_size, :img_size]
clean_mask = ((xx - 32)**2 + (yy - 32)**2 < 10**2).astype(int)
# Add random noise blobs (small isolated foreground pixels)
noise_mask = clean_mask.copy()
np.random.seed(0)
for _ in range(30):
    nr, nc = np.random.randint(0, img_size, 2)
    noise_mask[nr:nr+2, nc:nc+2] = 1  # 2x2 noise blob

eroded = erode(noise_mask, kernel_size=3)
dilated = dilate(noise_mask, kernel_size=3)
opened_mask = opening(noise_mask, kernel_size=3)
cleaned = remove_small_components(noise_mask, min_size=20)

fig, axes = plt.subplots(1, 5, figsize=(17, 3))
for ax, data, title in zip(
    axes,
    [clean_mask, noise_mask, eroded, opened_mask, cleaned],
    ['Ground Truth', 'Noisy Pred', 'Eroded', 'Opened', 'Remove small'],
):
    ax.imshow(data, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.suptitle('RW2: Morphological Post-Processing')
plt.tight_layout()
plt.savefig('/tmp/cv03_morphology.png', dpi=80)
plt.close()

print('Dice scores after post-processing:')
for mask, label in [(noise_mask,'Raw'), (eroded,'Eroded'), (opened_mask,'Opened'),
                    (cleaned,'Remove small')]:
    print(f'  {label:15s}: Dice = {dice_coefficient(mask, clean_mask):.3f}')


## Real-World Example 3: Multi-Class Segmentation

## Comparison: U-Net with vs without Skip Connections

In [ ]:
# ---- RW3 + Comparison: Multi-class IoU + skip connection ablation --------
# Multi-class segmentation assigns one of C classes to each pixel.
# Per-class IoU reveals which classes the model handles well vs poorly.


def generate_multiclass_masks(n=400, img_size=64, n_classes=3):
    """Generate 3-class segmentation data: background, circles, squares.

    Class 0: background
    Class 1: circles (filled discs)
    Class 2: squares (filled rectangles)
    Returns images (N,1,H,W) and masks (N,H,W) integer class labels.
    """
    images = np.zeros((n, 1, img_size, img_size), dtype=np.float32)
    masks = np.zeros((n, img_size, img_size), dtype=np.long)
    yy, xx = np.mgrid[:img_size, :img_size]
    for i in range(n):
        # Add a circle (class 1)
        cx, cy = np.random.randint(12, img_size-12, 2)
        r = np.random.randint(5, 12)
        circle = (xx - cx)**2 + (yy - cy)**2 < r**2
        images[i, 0][circle] = 0.8
        masks[i][circle] = 1
        # Add a square (class 2) that doesn't overlap circle
        attempts = 0
        while attempts < 20:
            sx, sy = np.random.randint(5, img_size-20, 2)
            sw, sh = np.random.randint(6, 14, 2)
            sq = np.zeros((img_size, img_size), dtype=bool)
            sq[sy:sy+sh, sx:sx+sw] = True
            if not np.any(sq & circle):  # no overlap
                images[i, 0][sq] = 0.5
                masks[i][sq] = 2
                break
            attempts += 1
        images[i, 0] += np.random.randn(img_size, img_size).astype(np.float32)*0.1
        images[i] = np.clip(images[i], 0, 1)
    return images, masks


X_mc, y_mc = generate_multiclass_masks(n=400)
X_mc_t = torch.tensor(X_mc)
y_mc_t = torch.tensor(y_mc, dtype=torch.long)
mc_loader = DataLoader(TensorDataset(X_mc_t[:320], y_mc_t[:320]),
                       batch_size=32, shuffle=True)
mc_val = TensorDataset(X_mc_t[320:], y_mc_t[320:])

# Multi-class U-Net: 3 output channels, one per class
mc_unet = UNet(in_ch=1, out_ch=3, base_ch=16).to(device)
mc_opt = optim.Adam(mc_unet.parameters(), lr=1e-3)
ce_crit = nn.CrossEntropyLoss()

for epoch in range(20):
    mc_unet.train()
    for X_b, y_b in mc_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        mc_opt.zero_grad()
        # CrossEntropyLoss expects (B, C, H, W) logits and (B, H, W) targets
        ce_crit(mc_unet(X_b), y_b).backward()
        mc_opt.step()

# Compute per-class IoU on validation set
mc_unet.eval()
all_preds, all_trues = [], []
with torch.no_grad():
    for X_v, y_v in DataLoader(mc_val, batch_size=32):
        pred_v = mc_unet(X_v.to(device)).argmax(1).cpu().numpy()  # (B, H, W)
        all_preds.append(pred_v.flatten())
        all_trues.append(y_v.numpy().flatten())

all_preds_np = np.concatenate(all_preds)
all_trues_np = np.concatenate(all_trues)

class_names = ['Background', 'Circles', 'Squares']
print('Per-class IoU:')
class_ious = []
for cls_id, cls_name in enumerate(class_names):
    cls_iou = jaccard_score(all_trues_np == cls_id, all_preds_np == cls_id)
    class_ious.append(cls_iou)
    print(f'  {cls_name:12s}: IoU = {cls_iou:.3f}')
print(f'  Mean IoU (mIoU): {np.mean(class_ious):.3f}')

# Comparison visualization: sample prediction
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
sample_img = X_mc_t[320:321].to(device)
sample_gt = y_mc_t[320].numpy()
with torch.no_grad():
    sample_pred = mc_unet(sample_img).argmax(1).cpu().numpy()[0]

axes[0].imshow(X_mc[320, 0], cmap='gray')
axes[0].set_title('Input Image')
axes[1].imshow(sample_gt, cmap='tab10', vmin=0, vmax=2)
axes[1].set_title('Ground Truth (3 classes)')
axes[2].imshow(sample_pred, cmap='tab10', vmin=0, vmax=2)
axes[2].set_title(f'Prediction (mIoU={np.mean(class_ious):.2f})')
for ax in axes:
    ax.axis('off')
plt.suptitle('RW3: Multi-Class Segmentation — Background / Circles / Squares')
plt.tight_layout()
plt.savefig('/tmp/cv03_multiclass.png', dpi=80)
plt.close()
print('Saved: /tmp/cv03_multiclass.png')
